# Import Bộ luật Hình sự → Neo4j

Dataset: `deepseek_merged.json`  
Thống kê: **2** Phần · **25** Chương · **411** Điều · **1326** Quy tắc · **3121** Điều kiện

## Schema
```
(Phan)-[:CO_CHUONG]->(Chuong)-[:CO_DIEU]->(DieuLuat)-[:CO_QUY_TAC]->(QuyTac)
                                   |                                      |
                            [:THUOC_NHOM]                    [:CO_DIEU_KIEN] / [:CO_HINH_PHAT]
                                   |                               |                  |
                              (NhomToi)                      (DieuKien)          (HinhPhat)
(DieuLuat)-[:LIEN_QUAN]->(DieuLuat)   (DieuKien)-[:LA_VAI_TRO]->(VaiTro)
```

## 1. Cài đặt thư viện

In [4]:
%pip install neo4j python-dotenv tqdm

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Cấu hình kết nối Neo4j

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

# Nạp chatbot/.env (cwd: chatbot hoặc gốc repo) và ưu tiên giá trị trong file này.
_workdir = Path.cwd()
_env_file = None
if (_workdir / ".env").is_file():
    _env_file = _workdir / ".env"
elif (_workdir / "chatbot" / ".env").is_file():
    _env_file = _workdir / "chatbot" / ".env"

if _env_file:
    load_dotenv(_env_file, override=True)
else:
    load_dotenv(override=True)

# Bắt buộc lấy cấu hình Neo4j từ .env để tránh vô tình dùng giá trị mặc định sai.
NEO4J_URI = os.getenv("NEO4J_URI")
NEO4J_USER = os.getenv("NEO4J_USER")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")

_missing_env = [
    name
    for name, value in {
        "NEO4J_URI": NEO4J_URI,
        "NEO4J_USER": NEO4J_USER,
        "NEO4J_PASSWORD": NEO4J_PASSWORD,
    }.items()
    if not value
]
if _missing_env:
    raise ValueError(f"Thiếu biến trong .env: {', '.join(_missing_env)}")

# Thư mục chatbot = thư mục chứa .env đã nạp
CHATBOT_ROOT = _env_file.parent if _env_file else _workdir
_for_dataset = CHATBOT_ROOT / "dataset" / "deepseek_merged.json"
DATASET_PATH = str(_for_dataset if _for_dataset.is_file() else CHATBOT_ROOT / "deepseek_merged.json")

print(f".env: {_env_file or '(không tìm thấy file, dùng biến hệ thống)'}")
print(f"URI: {NEO4J_URI}")
print(f"User: {NEO4J_USER}")
print(f"Dataset: {DATASET_PATH}")

.env: c:\Users\Admin\Desktop\DATN\chatbot\.env
URI: neo4j+s://cd9c5c25.databases.neo4j.io
User: cd9c5c25
Dataset: c:\Users\Admin\Desktop\DATN\chatbot\dataset\deepseek_merged.json


## 3. Kết nối & kiểm tra

In [6]:
from neo4j import GraphDatabase

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))

with driver.session() as session:
    result = session.run("RETURN 'Kết nối Neo4j thành công!' AS msg")
    print(result.single()["msg"])

Kết nối Neo4j thành công!


## 4. Load dataset

In [7]:
import json

with open(DATASET_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

parts = data["parts"]

# Thống kê
total_articles   = sum(len(c["articles"]) for p in parts for c in p["chapters"])
total_rules      = sum(len(a["rules"]) for p in parts for c in p["chapters"] for a in c["articles"])
total_conditions = sum(len(r["conditions"]) for p in parts for c in p["chapters"] for a in c["articles"] for r in a["rules"])
total_chapters   = sum(len(p["chapters"]) for p in parts)

print(f"✓ Phần       : {len(parts)}")
print(f"✓ Chương     : {total_chapters}")
print(f"✓ Điều luật  : {total_articles}")
print(f"✓ Quy tắc    : {total_rules}")
print(f"✓ Điều kiện  : {total_conditions}")

✓ Phần       : 2
✓ Chương     : 25
✓ Điều luật  : 411
✓ Quy tắc    : 1326
✓ Điều kiện  : 3121


## 5. Xoá dữ liệu cũ & tạo Constraints / Indexes

> ⚠️ Cell này xoá **toàn bộ** dữ liệu trong database. Bỏ qua nếu muốn giữ lại dữ liệu cũ.

In [8]:
SETUP_QUERIES = [
    # Xoá toàn bộ
    "MATCH (n) DETACH DELETE n",

    # Constraints (đảm bảo unique)
    "CREATE CONSTRAINT phan_id IF NOT EXISTS FOR (p:Phan)      REQUIRE p.part_id   IS UNIQUE",
    "CREATE CONSTRAINT chuong_id IF NOT EXISTS FOR (c:Chuong)  REQUIRE c.chapter_id IS UNIQUE",
    "CREATE CONSTRAINT dieu_id IF NOT EXISTS FOR (d:DieuLuat)  REQUIRE d.crime_id   IS UNIQUE",
    "CREATE CONSTRAINT rule_id IF NOT EXISTS FOR (r:QuyTac)    REQUIRE r.rule_id    IS UNIQUE",
    "CREATE CONSTRAINT nhom_id IF NOT EXISTS FOR (n:NhomToi)   REQUIRE n.ma_chuong  IS UNIQUE",
    "CREATE CONSTRAINT vaitro_id IF NOT EXISTS FOR (v:VaiTro)  REQUIRE v.ten        IS UNIQUE",

    # Indexes tăng tốc truy vấn
    "CREATE INDEX dieu_article IF NOT EXISTS FOR (d:DieuLuat) ON (d.article)",
    "CREATE INDEX quytac_logic IF NOT EXISTS FOR (r:QuyTac)   ON (r.logic)",
    "CREATE INDEX dk_type      IF NOT EXISTS FOR (k:DieuKien) ON (k.type)",

    # Full-text search cho điều kiện
    "CREATE FULLTEXT INDEX dk_text_search IF NOT EXISTS FOR (k:DieuKien) ON EACH [k.text]",
    "CREATE FULLTEXT INDEX dieu_name_search IF NOT EXISTS FOR (d:DieuLuat) ON EACH [d.name]",
]

with driver.session() as session:
    for q in SETUP_QUERIES:
        session.run(q)
        print(f"✓ {q[:70]}..." if len(q) > 70 else f"✓ {q}")

print("\n✅ Setup hoàn tất!")

✓ MATCH (n) DETACH DELETE n
✓ CREATE CONSTRAINT phan_id IF NOT EXISTS FOR (p:Phan)      REQUIRE p.pa...
✓ CREATE CONSTRAINT chuong_id IF NOT EXISTS FOR (c:Chuong)  REQUIRE c.ch...
✓ CREATE CONSTRAINT dieu_id IF NOT EXISTS FOR (d:DieuLuat)  REQUIRE d.cr...
✓ CREATE CONSTRAINT rule_id IF NOT EXISTS FOR (r:QuyTac)    REQUIRE r.ru...
✓ CREATE CONSTRAINT nhom_id IF NOT EXISTS FOR (n:NhomToi)   REQUIRE n.ma...
✓ CREATE CONSTRAINT vaitro_id IF NOT EXISTS FOR (v:VaiTro)  REQUIRE v.te...
✓ CREATE INDEX dieu_article IF NOT EXISTS FOR (d:DieuLuat) ON (d.article...
✓ CREATE INDEX quytac_logic IF NOT EXISTS FOR (r:QuyTac)   ON (r.logic)
✓ CREATE INDEX dk_type      IF NOT EXISTS FOR (k:DieuKien) ON (k.type)
✓ CREATE FULLTEXT INDEX dk_text_search IF NOT EXISTS FOR (k:DieuKien) ON...
✓ CREATE FULLTEXT INDEX dieu_name_search IF NOT EXISTS FOR (d:DieuLuat) ...

✅ Setup hoàn tất!


## 6. Import dữ liệu

### 6.1 Helper functions

In [9]:
import json
from tqdm import tqdm

# Các logic type liên quan đến vai trò đồng phạm
DONG_PHAM_LOGIC = {"ACCOMPLICE", "AGGREGATION"}

# Các condition type chứa thông tin vai trò
ROLE_CONDITION_TYPES = {"role", "actor", "subject"}

# Các condition type chứa tình tiết tăng/giảm nhẹ
TINH_TIET_TYPES = {"aggravating", "mitigating", "aggravating_circumstance"}

def _flatten_val(v):
    """
    Neo4j chỉ chấp nhận primitive (int, float, str, bool, None) hoặc list của primitive.
    Nếu v là dict  → serialize thành JSON string.
    Nếu v là list  → serialize thành JSON string (vì list of map cũng bị từ chối).
    Còn lại giữ nguyên.
    """
    if isinstance(v, (dict, list)):
        return json.dumps(v, ensure_ascii=False)
    return v

def build_hinh_phat_props(penalty: dict) -> dict:
    """
    Chuẩn hóa object hình phạt từ dataset.
    Một số field (fine, reform, prison, conditions) đôi khi là nested dict/list
    → flatten thành JSON string để Neo4j chấp nhận.

    Quy ước flatten:
      fine   dict  → fine_min, fine_max (int)   + fine_raw (JSON string)
      reform dict  → reform_min, reform_max, reform_unit + reform_raw
      prison dict  → prison_min, prison_max, prison_unit + prison_raw
      conditions list → conditions_raw (JSON string)
    """
    props = {}

    # ── Các field primitive đơn giản ─────────────────────────────────────────
    for key in ("min", "max", "extra", "note", "community_service",
                "deduction_rate", "probation"):
        props[key] = _flatten_val(penalty.get(key))

    # ── fine: có thể là int HOẶC dict {min, max} ─────────────────────────────
    fine = penalty.get("fine")
    if isinstance(fine, dict):
        props["fine_min"] = fine.get("min")
        props["fine_max"] = fine.get("max")
        props["fine_raw"] = json.dumps(fine, ensure_ascii=False)
        props["fine"]     = None   # giữ field để schema nhất quán
    else:
        props["fine"]     = fine
        props["fine_min"] = None
        props["fine_max"] = None
        props["fine_raw"] = None

    # ── reform: có thể là int HOẶC dict {min, max, unit} ────────────────────
    reform = penalty.get("reform")
    if isinstance(reform, dict):
        props["reform_min"]  = reform.get("min")
        props["reform_max"]  = reform.get("max")
        props["reform_unit"] = reform.get("unit")
        props["reform_raw"]  = json.dumps(reform, ensure_ascii=False)
        props["reform"]      = None
    else:
        props["reform"]      = reform
        props["reform_min"]  = None
        props["reform_max"]  = None
        props["reform_unit"] = None
        props["reform_raw"]  = None

    # ── prison: có thể là int HOẶC dict {min, max, unit} ────────────────────
    prison = penalty.get("prison")
    if isinstance(prison, dict):
        props["prison_min"]  = prison.get("min")
        props["prison_max"]  = prison.get("max")
        props["prison_unit"] = prison.get("unit")
        props["prison_raw"]  = json.dumps(prison, ensure_ascii=False)
        props["prison"]      = None
    else:
        props["prison"]      = prison
        props["prison_min"]  = None
        props["prison_max"]  = None
        props["prison_unit"] = None
        props["prison_raw"]  = None

    # ── conditions trong penalty (hiếm, serialize hết) ───────────────────────
    conds = penalty.get("conditions")
    props["conditions_raw"] = json.dumps(conds, ensure_ascii=False) if conds else None

    # Loại bỏ key có value None để tránh ghi null thừa vào Neo4j
    return {k: v for k, v in props.items() if v is not None}

# ── Smoke test ────────────────────────────────────────────────────────────────
test_cases = [
    {"min": 3, "max": 7, "fine": 1000000},
    {"min": 3, "max": 7, "fine": {"min": 100000000, "max": 500000000}},
    {"min": 3, "max": 12, "reform": {"min": 0, "max": 2, "unit": "year"},
     "prison": {"min": 3, "max": 6, "unit": "month"},
     "conditions": [{"type": "condition", "text": "Gây thiệt hại..."}]},
]
print("Smoke test build_hinh_phat_props:")
for tc in test_cases:
    result = build_hinh_phat_props(tc)
    # Kiểm tra KHÔNG có nested dict/list
    bad = {k: v for k, v in result.items() if isinstance(v, (dict, list))}
    status = '✅ OK' if not bad else f'❌ VẪN CÒN nested: {bad}'
    print(f"  {status} → {result}")
print("\n✓ Helper functions đã sẵn sàng")

Smoke test build_hinh_phat_props:
  ✅ OK → {'min': 3, 'max': 7, 'fine': 1000000}
  ✅ OK → {'min': 3, 'max': 7, 'fine_min': 100000000, 'fine_max': 500000000, 'fine_raw': '{"min": 100000000, "max": 500000000}'}
  ✅ OK → {'min': 3, 'max': 12, 'reform_min': 0, 'reform_max': 2, 'reform_unit': 'year', 'reform_raw': '{"min": 0, "max": 2, "unit": "year"}', 'prison_min': 3, 'prison_max': 6, 'prison_unit': 'month', 'prison_raw': '{"min": 3, "max": 6, "unit": "month"}', 'conditions_raw': '[{"type": "condition", "text": "Gây thiệt hại..."}]'}

✓ Helper functions đã sẵn sàng


### 6.2 Import Phần & Chương

In [10]:
def import_phan_chuong(tx, part_data):
    part = part_data["part"]

    # Tạo node Phan
    tx.run("""
        MERGE (p:Phan {part_id: $part_id})
        SET p.name = $name
    """, part_id=part["part_id"], name=part["name"])

    for chap in part_data["chapters"]:
        # Tạo node Chuong
        tx.run("""
            MERGE (c:Chuong {chapter_id: $chapter_id})
            SET c.name = $name,
                c.part_id = $part_id
        """, chapter_id=chap["chapter_id"],
             name=chap["name"],
             part_id=part["part_id"])

        # Quan hệ Phan -> Chuong
        tx.run("""
            MATCH (p:Phan {part_id: $part_id})
            MATCH (c:Chuong {chapter_id: $chapter_id})
            MERGE (p)-[:CO_CHUONG]->(c)
        """, part_id=part["part_id"], chapter_id=chap["chapter_id"])

        # Tạo node NhomToi (từ tên chương)
        tx.run("""
            MERGE (n:NhomToi {ma_chuong: $chapter_id})
            SET n.ten = $name
        """, chapter_id=chap["chapter_id"], name=chap["name"])

        # Quan hệ Chuong -> NhomToi
        tx.run("""
            MATCH (c:Chuong  {chapter_id: $chapter_id})
            MATCH (n:NhomToi {ma_chuong:  $chapter_id})
            MERGE (c)-[:THUOC_NHOM]->(n)
        """, chapter_id=chap["chapter_id"])

with driver.session() as session:
    for part_data in tqdm(parts, desc="Phan & Chuong"):
        session.execute_write(import_phan_chuong, part_data)

print("✅ Import Phan & Chuong hoàn tất")

Phan & Chuong: 100%|██████████| 2/2 [00:06<00:00,  3.27s/it]

✅ Import Phan & Chuong hoàn tất


### 6.3 Import DieuLuat

In [11]:
def import_dieu_luat(tx, chapter_id, article_data):
    crime = article_data["crime"]

    tx.run("""
        MERGE (d:DieuLuat {crime_id: $crime_id})
        SET d.name       = $name,
            d.article    = $article,
            d.chapter_id = $chapter_id
    """, crime_id=str(crime["id"]),
         name=crime["name"],
         article=crime["article"],
         chapter_id=chapter_id)

    # Quan hệ Chuong -> DieuLuat
    tx.run("""
        MATCH (c:Chuong   {chapter_id: $chapter_id})
        MATCH (d:DieuLuat {crime_id:   $crime_id})
        MERGE (c)-[:CO_DIEU]->(d)
    """, chapter_id=chapter_id, crime_id=str(crime["id"]))

with driver.session() as session:
    all_articles = [
        (chap["chapter_id"], art)
        for part in parts
        for chap in part["chapters"]
        for art in chap["articles"]
    ]
    for chapter_id, art in tqdm(all_articles, desc="DieuLuat"):
        session.execute_write(import_dieu_luat, chapter_id, art)

print("✅ Import DieuLuat hoàn tất")

DieuLuat: 100%|██████████| 411/411 [02:36<00:00,  2.63it/s]

✅ Import DieuLuat hoàn tất


### 6.4 Import QuyTac + DieuKien + HinhPhat

In [12]:
def import_quy_tac(tx, crime_id, rule):
    rule_id = rule["rule_id"]

    # ── Node QuyTac ──────────────────────────────────────────────────────────
    tx.run("""
        MERGE (r:QuyTac {rule_id: $rule_id})
        SET r.clause   = $clause,
            r.logic    = $logic,
            r.priority = $priority,
            r.crime_id = $crime_id
    """, rule_id=rule_id,
         clause=rule.get("clause"),
         logic=rule.get("logic", ""),
         priority=rule.get("priority", 1),
         crime_id=crime_id)

    # Quan hệ DieuLuat -> QuyTac
    tx.run("""
        MATCH (d:DieuLuat {crime_id: $crime_id})
        MATCH (r:QuyTac   {rule_id:  $rule_id})
        MERGE (d)-[:CO_QUY_TAC]->(r)
        
    """, crime_id=crime_id, rule_id=rule_id)

    # ── Node HinhPhat ─────────────────────────────────────────────────────────
    penalty = rule.get("penalty", {})
    if penalty:
        hp_id = f"hp_{rule_id}"
        hp_props = build_hinh_phat_props(penalty)
        hp_props["hp_id"] = hp_id
        hp_props["rule_id"] = rule_id

        tx.run("""
            MERGE (hp:HinhPhat {hp_id: $hp_id})
            SET hp += $props
        """, hp_id=hp_id, props=hp_props)

        tx.run("""
            MATCH (r:QuyTac   {rule_id: $rule_id})
            MATCH (hp:HinhPhat {hp_id:  $hp_id})
            MERGE (r)-[:CO_HINH_PHAT]->(hp)
        """, rule_id=rule_id, hp_id=hp_id)

    # ── Nodes DieuKien ────────────────────────────────────────────────────────
    for i, cond in enumerate(rule.get("conditions", [])):
        dk_id = f"dk_{rule_id}_{i}"
        cond_type = cond.get("type", "other")

        tx.run("""
            MERGE (dk:DieuKien {dk_id: $dk_id})
            SET dk.type    = $type,
                dk.text    = $text,
                dk.point   = $point,
                dk.rule_id = $rule_id
        """, dk_id=dk_id,
             type=cond_type,
             text=cond.get("text", ""),
             point=cond.get("point"),
             rule_id=rule_id)

        tx.run("""
            MATCH (r:QuyTac  {rule_id: $rule_id})
            MATCH (dk:DieuKien {dk_id: $dk_id})
            MERGE (r)-[:CO_DIEU_KIEN]->(dk)
        """, rule_id=rule_id, dk_id=dk_id)

        # ── Node VaiTro (nếu condition chứa thông tin vai trò) ────────────────
        if cond_type in ROLE_CONDITION_TYPES and cond.get("text"):
            vai_tro_text = cond["text"].strip()
            tx.run("""
                MERGE (v:VaiTro {ten: $ten})
                ON CREATE SET v.loai = $loai
            """, ten=vai_tro_text,
                 loai="dong_pham" if rule.get("logic") in DONG_PHAM_LOGIC else "chinh_pham")

            tx.run("""
                MATCH (dk:DieuKien {dk_id: $dk_id})
                MATCH (v:VaiTro    {ten:   $ten})
                MERGE (dk)-[:LA_VAI_TRO]->(v)
            """, dk_id=dk_id, ten=vai_tro_text)

        # ── Node TinhTiet (nếu là tình tiết tăng/giảm nhẹ) ───────────────────
        if cond_type in TINH_TIET_TYPES and cond.get("text"):
            loai_tt = "tang_nang" if "aggravating" in cond_type else "giam_nhe"
            tt_id = f"tt_{dk_id}"
            tx.run("""
                MERGE (tt:TinhTiet {tt_id: $tt_id})
                SET tt.loai      = $loai,
                    tt.noi_dung  = $noi_dung
            """, tt_id=tt_id,
                 loai=loai_tt,
                 noi_dung=cond.get("text", ""))

            tx.run("""
                MATCH (dk:DieuKien {dk_id: $dk_id})
                MATCH (tt:TinhTiet {tt_id: $tt_id})
                MERGE (dk)-[:LA_TINH_TIET]->(tt)
            """, dk_id=dk_id, tt_id=tt_id)


with driver.session() as session:
    all_rules = [
        (str(art["crime"]["id"]), rule)
        for part in parts
        for chap in part["chapters"]
        for art in chap["articles"]
        for rule in art["rules"]
    ]
    for crime_id, rule in tqdm(all_rules, desc="QuyTac + DieuKien + HinhPhat"):
        session.execute_write(import_quy_tac, crime_id, rule)

print("✅ Import QuyTac + DieuKien + HinhPhat hoàn tất")

QuyTac + DieuKien + HinhPhat: 100%|██████████| 1326/1326 [29:24<00:00,  1.33s/it] 

✅ Import QuyTac + DieuKien + HinhPhat hoàn tất


### 6.5 Tạo quan hệ LIEN_QUAN giữa các DieuLuat

Phát hiện các điều luật tham chiếu chéo nhau qua text trong DieuKien.

In [13]:
import re

def extract_article_refs(text: str) -> list[int]:
    """Trích xuất số điều luật được đề cập trong text.
    VD: 'theo quy định tại Điều 168' -> [168]
    """
    patterns = [
        r'[Đđ]iều\s+(\d+)',
        r'[Aa]rticle\s+(\d+)',
        r'khoản\s+\d+\s+[Đđ]iều\s+(\d+)',
    ]
    refs = []
    for pat in patterns:
        refs.extend(int(m) for m in re.findall(pat, text))
    return list(set(refs))


def create_lien_quan(tx, from_crime_id: str, to_article: int, ly_do: str):
    tx.run("""
        MATCH (d1:DieuLuat {crime_id: $from_id})
        MATCH (d2:DieuLuat {article:  $to_article})
        WHERE d1 <> d2
        MERGE (d1)-[r:LIEN_QUAN]->(d2)
        ON CREATE SET r.ly_do = $ly_do
        ON MATCH  SET r.ly_do = $ly_do
    """, from_id=from_crime_id,
         to_article=to_article,
         ly_do=ly_do)


lien_quan_count = 0

with driver.session() as session:
    for part in tqdm(parts, desc="LIEN_QUAN"):
        for chap in part["chapters"]:
            for art in chap["articles"]:
                crime_id = str(art["crime"]["id"])
                for rule in art["rules"]:
                    for cond in rule.get("conditions", []):
                        text = cond.get("text", "")
                        if not text:
                            continue
                        refs = extract_article_refs(text)
                        for ref_article in refs:
                            if ref_article != art["crime"]["article"]:
                                session.execute_write(
                                    create_lien_quan,
                                    crime_id, ref_article,
                                    cond.get("type", "reference")
                                )
                                lien_quan_count += 1

print(f"✅ Tạo {lien_quan_count} quan hệ LIEN_QUAN")

LIEN_QUAN: 100%|██████████| 2/2 [00:39<00:00, 19.96s/it]

✅ Tạo 129 quan hệ LIEN_QUAN


## 7. Kiểm tra kết quả import

In [14]:
CHECK_QUERIES = {
    "Phan"      : "MATCH (n:Phan)      RETURN count(n) AS cnt",
    "Chuong"    : "MATCH (n:Chuong)    RETURN count(n) AS cnt",
    "NhomToi"   : "MATCH (n:NhomToi)   RETURN count(n) AS cnt",
    "DieuLuat"  : "MATCH (n:DieuLuat)  RETURN count(n) AS cnt",
    "QuyTac"    : "MATCH (n:QuyTac)    RETURN count(n) AS cnt",
    "DieuKien"  : "MATCH (n:DieuKien)  RETURN count(n) AS cnt",
    "HinhPhat"  : "MATCH (n:HinhPhat)  RETURN count(n) AS cnt",
    "VaiTro"    : "MATCH (n:VaiTro)    RETURN count(n) AS cnt",
    "TinhTiet"  : "MATCH (n:TinhTiet)  RETURN count(n) AS cnt",
    "CO_CHUONG" : "MATCH ()-[r:CO_CHUONG]->()    RETURN count(r) AS cnt",
    "CO_DIEU"   : "MATCH ()-[r:CO_DIEU]->()      RETURN count(r) AS cnt",
    "CO_QUY_TAC": "MATCH ()-[r:CO_QUY_TAC]->()   RETURN count(r) AS cnt",
    "CO_HINH_PHAT"  : "MATCH ()-[r:CO_HINH_PHAT]->()   RETURN count(r) AS cnt",
    "CO_DIEU_KIEN"  : "MATCH ()-[r:CO_DIEU_KIEN]->()   RETURN count(r) AS cnt",
    "LIEN_QUAN"     : "MATCH ()-[r:LIEN_QUAN]->()       RETURN count(r) AS cnt",
    "LA_VAI_TRO"    : "MATCH ()-[r:LA_VAI_TRO]->()      RETURN count(r) AS cnt",
    "LA_TINH_TIET"  : "MATCH ()-[r:LA_TINH_TIET]->()    RETURN count(r) AS cnt",
}

with driver.session() as session:
    print(f"{'Loại':<20} {'Số lượng':>10}")
    print("-" * 32)
    for label, q in CHECK_QUERIES.items():
        cnt = session.run(q).single()["cnt"]
        print(f"{label:<20} {cnt:>10,}")

Loại                   Số lượng
--------------------------------
Phan                          2
Chuong                       25
NhomToi                      25
DieuLuat                    411
QuyTac                    1,326
DieuKien                  3,121
HinhPhat                  1,326
VaiTro                        9
TinhTiet                  1,873
CO_CHUONG                    25
CO_DIEU                     411
CO_QUY_TAC                1,326
CO_HINH_PHAT              1,326
CO_DIEU_KIEN              3,121
LIEN_QUAN                   118
LA_VAI_TRO                   13
LA_TINH_TIET              1,873


## 8. Truy vấn mẫu cho RAG

### 8.1 Tìm tội danh theo hành vi (full-text search)

In [15]:
HANH_VI = "cướp tài sản"  # ← thay bằng hành vi cần tra cứu

query = """
CALL db.index.fulltext.queryNodes('dieu_name_search', $keyword)
YIELD node AS d, score
MATCH (d)-[:CO_QUY_TAC]->(r:QuyTac {logic: 'BASE'})
MATCH (r)-[:CO_HINH_PHAT]->(hp:HinhPhat)
RETURN d.article  AS dieu,
       d.name     AS ten_toi,
       r.clause   AS khoan,
       hp.min     AS min_nam,
       hp.max     AS max_nam,
       hp.extra   AS hinh_phat_them,
       score
ORDER BY score DESC
LIMIT 5
"""

with driver.session() as session:
    results = session.run(query, keyword=HANH_VI)
    for r in results:
        print(f"Điều {r['dieu']}: {r['ten_toi']}")
        print(f"  Khoản {r['khoan']}: {r['min_nam']} - {r['max_nam']} năm {r['hinh_phat_them'] or ''}")
        print(f"  Relevance score: {r['score']:.3f}\n")

Điều 168: Tội cướp tài sản
  Khoản 1: 3 - 10 năm 
  Relevance score: 5.878

Điều 171: Tội cướp giật tài sản
  Khoản 1: 1 - 5 năm 
  Relevance score: 5.597

Điều 302: Tội cướp biển
  Khoản 1: 5 - 10 năm 
  Relevance score: 3.086

Điều 353: Tội tham ô tài sản
  Khoản 1: None - None năm 
  Relevance score: 2.807

Điều 173: Tội trộm cắp tài sản
  Khoản 1: None - None năm 
  Relevance score: 2.807



### 8.2 Phân tích vai trò đồng phạm

In [16]:
DIEU_LUAT = 168  # ← Điều cần tra (VD: 168 = Tội cướp)

query = """
MATCH (d:DieuLuat {article: $article})
MATCH (d)-[:CO_QUY_TAC]->(r:QuyTac)
MATCH (r)-[:CO_DIEU_KIEN]->(dk:DieuKien)
WHERE dk.type IN ['role', 'actor', 'subject']
MATCH (r)-[:CO_HINH_PHAT]->(hp:HinhPhat)
RETURN d.name       AS ten_toi,
       r.logic      AS logic_type,
       r.clause     AS khoan,
       dk.text      AS vai_tro,
       hp.min       AS min_nam,
       hp.max       AS max_nam,
       hp.extra     AS them
ORDER BY r.priority
"""

with driver.session() as session:
    results = session.run(query, article=DIEU_LUAT)
    for r in results:
        print(f"[{r['logic_type']}] Khoản {r['khoan']}: {r['vai_tro']}")
        print(f"  → {r['min_nam']} - {r['max_nam']} năm {r['them'] or ''}\n")

### 8.3 Tình tiết tăng nặng / giảm nhẹ

In [17]:
DIEU_LUAT = 168

query = """
MATCH (d:DieuLuat {article: $article})
MATCH (d)-[:CO_QUY_TAC]->(r:QuyTac)
WHERE r.logic IN ['AGGRAVATION', 'MITIGATION']
MATCH (r)-[:CO_DIEU_KIEN]->(dk:DieuKien)
MATCH (r)-[:CO_HINH_PHAT]->(hp:HinhPhat)
RETURN r.logic  AS loai,
       dk.text  AS tinh_tiet,
       hp.min   AS min_nam,
       hp.max   AS max_nam
ORDER BY loai
"""

with driver.session() as session:
    results = session.run(query, article=DIEU_LUAT)
    current = None
    for r in results:
        if r["loai"] != current:
            current = r["loai"]
            print(f"\n{'🔺 TĂNG NẶNG' if 'AGGRAV' in current else '🔻 GIẢM NHẸ'}")
        print(f"  {r['tinh_tiet']}: {r['min_nam']}-{r['max_nam']} năm")


🔺 TĂNG NẶNG
  Có tổ chức: 7-15 năm
  Có tính chất chuyên nghiệp: 7-15 năm
  Gây thương tích hoặc gây tổn hại cho sức khỏe của người khác mà tỷ lệ tổn thương cơ thể từ 11% đến 30%: 7-15 năm
  Sử dụng vũ khí, phương tiện hoặc thủ đoạn nguy hiểm khác: 7-15 năm
  Chiếm đoạt tài sản trị giá từ 50.000.000 đồng đến dưới 200.000.000 đồng: 7-15 năm
  Phạm tội đối với người dưới 16 tuổi, phụ nữ mà biết là có thai, người già yếu hoặc người không có khả năng tự vệ: 7-15 năm
  Gây ảnh hưởng xấu đến an ninh, trật tự, an toàn xã hội: 7-15 năm
  Tái phạm nguy hiểm: 7-15 năm
  Chiếm đoạt tài sản trị giá từ 200.000.000 đồng đến dưới 500.000.000 đồng: 12-20 năm
  Gây thương tích hoặc gây tổn hại cho sức khỏe của người khác mà tỷ lệ tổn thương cơ thể từ 31% đến 60%: 12-20 năm
  Lợi dụng thiên tai, dịch bệnh: 12-20 năm
  Chiếm đoạt tài sản trị giá 500.000.000 đồng trở lên: 18-20 năm
  Gây thương tích hoặc gây tổn hại cho sức khỏe của 01 người mà tỷ lệ tổn thương cơ thể 61% trở lên hoặc gây thương tích hoặ

### 8.4 Truy vấn tổng hợp cho RAG pipeline

Query này được dùng để lấy full context khi người dùng mô tả tình huống.

In [20]:
def get_full_context(article_number: int) -> dict:
    """
    Lấy toàn bộ context của một điều luật để đưa vào LLM.

    Lưu ý schema HinhPhat sau khi flatten:
      hp.fine   → có thể None (nếu là dict thì đã tách thành hp.fine_min / hp.fine_max)
      hp.reform → có thể None (tách thành hp.reform_min / hp.reform_max / hp.reform_unit)
      hp.prison → có thể None (tách thành hp.prison_min / hp.prison_max / hp.prison_unit)
    """
    query = """
    MATCH (d:DieuLuat {article: $article})
    OPTIONAL MATCH (d)<-[:CO_DIEU]-(c:Chuong)
    OPTIONAL MATCH (c)-[:THUOC_NHOM]->(n:NhomToi)

    MATCH (d)-[:CO_QUY_TAC]->(r:QuyTac)
    OPTIONAL MATCH (r)-[:CO_HINH_PHAT]->(hp:HinhPhat)
    OPTIONAL MATCH (r)-[:CO_DIEU_KIEN]->(dk:DieuKien)
    OPTIONAL MATCH (d)-[:LIEN_QUAN]->(d2:DieuLuat)

    RETURN
        d.article AS dieu,
        d.name    AS ten_toi,
        n.ten     AS nhom_toi,
        c.name    AS chuong,

        // Quy tắc + hình phạt — dùng đúng tên field sau khi flatten
        collect(DISTINCT {
            khoan    : r.clause,
            logic    : r.logic,
            priority : r.priority,
            hinh_phat: {
                min        : hp.min,
                max        : hp.max,
                extra      : hp.extra,
                note       : hp.note,
                fine       : hp.fine,
                fine_min   : hp.fine_min,
                fine_max   : hp.fine_max,
                reform_min : hp.reform_min,
                reform_max : hp.reform_max,
                reform_unit: hp.reform_unit,
                prison_min : hp.prison_min,
                prison_max : hp.prison_max,
                prison_unit: hp.prison_unit
            }
        }) AS quy_tac,

        collect(DISTINCT {
            type: dk.type,
            text: dk.text
        }) AS dieu_kien,

        collect(DISTINCT d2.article) AS dieu_lien_quan
    """

    with driver.session() as session:
        record = session.run(query, article=article_number).single()
        if not record:
            return {}
        ctx = dict(record)

        # Loại bỏ None khỏi hinh_phat để output gọn hơn
        ctx["quy_tac"] = [
            {
                **{k: v for k, v in qt.items() if k != "hinh_phat"},
                "hinh_phat": {k: v for k, v in qt["hinh_phat"].items() if v is not None}
            }
            for qt in ctx["quy_tac"]
        ]
        return ctx


def format_hinh_phat(hp: dict) -> str:
    """Chuyển dict hình phạt thành chuỗi dễ đọc."""
    parts = []
    if hp.get("min") is not None and hp.get("max") is not None:
        parts.append(f"{hp['min']}–{hp['max']} năm tù")
    if hp.get("extra"):
        parts.append(hp["extra"])
    if hp.get("fine") is not None:
        parts.append(f"phạt tiền {hp['fine']:,} đồng")
    if hp.get("fine_min") is not None:
        parts.append(f"phạt tiền {hp['fine_min']:,}–{hp['fine_max']:,} đồng")
    if hp.get("reform_min") is not None:
        unit = hp.get('reform_unit', 'năm')
        parts.append(f"cải tạo không giam giữ {hp['reform_min']}–{hp['reform_max']} {unit}")
    if hp.get("prison_min") is not None:
        unit = hp.get('prison_unit', 'tháng')
        parts.append(f"tù {hp['prison_min']}–{hp['prison_max']} {unit}")
    if hp.get("note"):
        parts.append(hp["note"])
    return "; ".join(parts) if parts else "(xem thêm)"


# ── Test ─────────────────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings("ignore")  # tắt WARNING về property không tồn tại

ctx = get_full_context(168)
if ctx:
    print(f"Điều {ctx['dieu']}: {ctx['ten_toi']}")
    print(f"Chương     : {ctx['chuong']}")
    print(f"Nhóm tội   : {ctx['nhom_toi']}")
    print(f"Điều liên quan: {ctx['dieu_lien_quan']}")
    print(f"\nChi tiết {len(ctx['quy_tac'])} quy tắc:")
    for qt in sorted(ctx['quy_tac'], key=lambda x: x.get('priority', 0)):
        hp_str = format_hinh_phat(qt.get('hinh_phat', {}))
        print(f"  [{qt['logic']}] Khoản {qt['khoan']}: {hp_str}")
else:
    print("Không tìm thấy điều luật này trong database")

Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.UnknownPropertyKeyWarning} {category: UNRECOGNIZED} {title: The provided property key is not in the database} {description: One of the property names in your query is not available in the database, make sure you didn't misspell it or that the label is available when you run this statement in your application (the missing property name is: fine)} {position: line: 27, column: 33, offset: 874} for query: '\n    MATCH (d:DieuLuat {article: $article})\n    OPTIONAL MATCH (d)<-[:CO_DIEU]-(c:Chuong)\n    OPTIONAL MATCH (c)-[:THUOC_NHOM]->(n:NhomToi)\n\n    MATCH (d)-[:CO_QUY_TAC]->(r:QuyTac)\n    OPTIONAL MATCH (r)-[:CO_HINH_PHAT]->(hp:HinhPhat)\n    OPTIONAL MATCH (r)-[:CO_DIEU_KIEN]->(dk:DieuKien)\n    OPTIONAL MATCH (d)-[:LIEN_QUAN]->(d2:DieuLuat)\n\n    RETURN\n        d.article AS dieu,\n        d.name    AS ten_toi,\n        n.ten     AS nhom_toi,\n        c.name    AS chuong,\n\n        

Điều 168: Tội cướp tài sản
Chương     : Các tội xâm phạm sở hữu
Nhóm tội   : Các tội xâm phạm sở hữu
Điều liên quan: []

Chi tiết 6 quy tắc:
  [BASE] Khoản 1: 3–10 năm tù
  [AGGRAVATION] Khoản 2: 7–15 năm tù
  [AGGRAVATION] Khoản 3: 12–20 năm tù
  [AGGRAVATION] Khoản 4: 18–20 năm tù; tù chung thân
  [PREPARATION] Khoản 5: 1–5 năm tù
  [ADDITIONAL_PENALTY] Khoản 6: Có thể bị phạt tiền từ 10.000.000 đồng đến 100.000.000 đồng, phạt quản chế, cấm cư trú từ 01 năm đến 05 năm hoặc tịch thu một phần hoặc toàn bộ tài sản


## 9. Đóng kết nối

In [21]:
driver.close()
print("✅ Hoàn tất! Đã đóng kết nối Neo4j.")
print("")
print("Tóm tắt những gì đã import:")
print("  • Node: Phan, Chuong, NhomToi, DieuLuat, QuyTac, DieuKien, HinhPhat, VaiTro, TinhTiet")
print("  • Relationship: CO_CHUONG, CO_DIEU, THUOC_NHOM, CO_QUY_TAC,")
print("                  CO_HINH_PHAT, CO_DIEU_KIEN, LIEN_QUAN, LA_VAI_TRO, LA_TINH_TIET")
print("  • Index: fulltext trên DieuKien.text và DieuLuat.name")

✅ Hoàn tất! Đã đóng kết nối Neo4j.

Tóm tắt những gì đã import:
  • Node: Phan, Chuong, NhomToi, DieuLuat, QuyTac, DieuKien, HinhPhat, VaiTro, TinhTiet
  • Relationship: CO_CHUONG, CO_DIEU, THUOC_NHOM, CO_QUY_TAC,
                  CO_HINH_PHAT, CO_DIEU_KIEN, LIEN_QUAN, LA_VAI_TRO, LA_TINH_TIET
  • Index: fulltext trên DieuKien.text và DieuLuat.name
